In [1]:
import psycopg
import numpy as np
from pgvector.psycopg import register_vector
from sentence_transformers import SentenceTransformer

# 1. Load the lightweight 384-dimensional local embedding model
print("Loading local AI model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Connect to your local Postgres database
conn = psycopg.connect("postgresql://shilpisharma@localhost:5432/postgres")

with conn.cursor() as cur:
    # Enable extension and create the 384-D schema
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS sample_384_docs (
            id serial PRIMARY KEY,
            content text,
            embedding vector(384)
        );
    """)
    # Add an HNSW index optimized for Cosine Distance matching
    cur.execute("CREATE INDEX IF NOT EXISTS idx_hnsw_384 ON sample_384_docs USING hnsw (embedding vector_cosine_ops);")
    conn.commit()

# Register type mapping with psycopg
register_vector(conn)
print("Database schema initialized and HNSW Index created successfully!")

Loading local AI model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Database schema initialized and HNSW Index created successfully!


In [2]:
# Sample text sentences to vectorize
documents = [
    "PostgreSQL is a powerful, open-source object-relational database system.",
    "Artificial intelligence and machine learning are transforming modern software development.",
    "An index allows database engines to look up rows without running a full sequential scan.",
    "The perfect recipe for Italian pasta requires fresh eggs, semolina flour, and salt."
]

with conn.cursor() as cur:
    for doc in documents:
        # Convert text string into a 384-dimensional numpy array natively
        vector_embedding = model.encode(doc)
        
        cur.execute(
            "INSERT INTO sample_384_docs (content, embedding) VALUES (%s, %s);",
            (doc, vector_embedding)
        )
    conn.commit()

print(f"Successfully processed and inserted {len(documents)} text embeddings!")

Successfully processed and inserted 4 text embeddings!


In [3]:
# Our search phrase doesn't contain the words "PostgreSQL", "index", or "scan"
user_search_query = "How do databases search for tables quickly?"

# 1. Convert the search query into the exact same 384-D vector space
query_vector = model.encode(user_search_query)

with conn.cursor() as cur:
    # 2. Query using Cosine Distance (<=>)
    cur.execute("""
        SELECT content, embedding <=> %s AS distance 
        FROM sample_384_docs 
        ORDER BY distance ASC 
        LIMIT 2;
    """, (query_vector,))
    
    print(f"Search Query: '{user_search_query}'\n")
    print("Top Semantic Matches:")
    print("-" * 40)
    for content, distance in cur.fetchall():
        # A smaller cosine distance score means the meanings are closer together
        print(f"Score (Distance): {distance:.4f} | Text: {content}")

# Always remember to close your connection cleanly when done
conn.close()

Search Query: 'How do databases search for tables quickly?'

Top Semantic Matches:
----------------------------------------
Score (Distance): 0.4153 | Text: An index allows database engines to look up rows without running a full sequential scan.
Score (Distance): 0.4153 | Text: An index allows database engines to look up rows without running a full sequential scan.
